# NeurIPS 2026 replication, notebook 6: Paper A features under the recovered timm protocol (GPU)
Runtime: GPU. Re-extracts the 16 Paper A configurations (ViT-B/16-IN21K, DINOv2-B/14, CLIP-B/16, random ViT-B/16 on CIFAR-100, ImageNet-R, CUB-200, OmniBenchmark) under the protocol the archaeology recovered: `timm.create_model(name, pretrained=True, num_classes=0, img_size=224)` (the pooled pre-logits tensor), CIFAR transform `Resize(224)` with CIFAR-100 statistics, CUB transform `Resize(256) + CenterCrop(224)` with ImageNet statistics (ImageNet-R and OmniBenchmark use the CUB transform until the archaeology says otherwise; the choice is recorded in every cache's meta). Caches go to a SEPARATE Drive root `features_timm/`; the existing `features/` caches are not touched. The last cell runs the fingerprint readback against the printed table and applies the decision rules of `FINGERPRINT_DECISION_TREE.md`.

In [ ]:
# ---- 0. Setup: Drive, repo, packages ---------------------------------------------
# Run this cell first in every notebook. Session state is lost between sessions;
# the feature caches on Drive are not.
from google.colab import drive; drive.mount('/content/drive')
import os, subprocess, sys, pathlib
WORK = pathlib.Path('/content/drive/MyDrive/neurips2026_replication'); WORK.mkdir(parents=True, exist_ok=True)
REPO_ZIP = WORK / 'repo.zip'            # upload the repo zip here once (claude_replication/repo.zip), or set GIT_URL
GIT_URL = ''                            # e.g. 'https://github.com/<owner>/<repo>.git' once the repo is public
if GIT_URL:
    subprocess.run(['git','clone','--depth','1',GIT_URL,'/content/repo'], check=True)
else:
    assert REPO_ZIP.exists(), f'upload the repo zip to {REPO_ZIP}'
    subprocess.run(['rm','-rf','/content/repo','/content/repo_unz']); subprocess.run(['unzip','-q','-o',str(REPO_ZIP),'-d','/content/repo_unz'], check=True)
    inner = next(p for p in pathlib.Path('/content/repo_unz').rglob('replicate.py')).parent
    subprocess.run(['mv', str(inner), '/content/repo'], check=True)
os.chdir('/content/repo'); sys.path.insert(0, '/content/repo')
os.environ['REPLICATION_FEATURES'] = str(WORK / 'features_timm')   # the SEPARATE cache root of the recovered protocol
os.environ['REPLICATION_ALLOW_DOWNLOADS'] = '1'   # this notebook runs on Colab only; the subprocess below inherits the download permission (pipeline/features.py::_assert_download_allowed)
subprocess.run([sys.executable,'-m','pip','install','-q','timm>=1.0','torchvision','gdown','datasets','scipy','scikit-learn','matplotlib'], check=True)
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('features ->', os.environ['REPLICATION_FEATURES'])

In [ ]:
# ---- Configuration: replace the timm names with the ones the owner's stage_a_features.py uses, if they differ ----
from pipeline import features_timm as FT
TIMM_IDS = {
    'vit_in21k':  dict(name='vit_base_patch16_224.augreg_in21k', pretrained=True),   # the original IN21K ViT is 'vit_base_patch16_224.orig_in21k' in timm
    'dinov2_b14': dict(name='vit_base_patch14_dinov2.lvd142m', pretrained=True),     # img_size=224 resamples the position embedding; '..._reg4_dinov2.lvd142m' is the registers variant (round 3)
    'clip_b16':   dict(name='vit_base_patch16_clip_224.openai', pretrained=True),    # pooled pre-logits, 768-d, pre-projection; '.laion2b' is the LAION variant (round 3)
    'random_vit': dict(name='vit_base_patch16_224', pretrained=False, seed=0),       # random init, torch seed 0
}
FT.TIMM_IDS.update(TIMM_IDS)
BACKBONES = ['vit_in21k', 'dinov2_b14', 'clip_b16', 'random_vit']
DATASETS = ['cifar100', 'imagenet_r', 'cub200', 'omnibenchmark']
DATA_ROOT = '/content/data'               # the same raw images as notebook 01 (downloaded again if the session is fresh)
OUT_ROOT = os.environ['REPLICATION_FEATURES']
FP16 = False                               # keep full precision for the fingerprint comparison
TRANSFORM = 'timm'                         # 'timm': each model's own eval transform (bicubic 248 -> centre crop 224, the checkpoint's statistics);
                                           #         this is the convention under which the fleet's round 2 recovered the printed ViT values.
                                           # 'archaeology': the per-dataset transforms the archaeology named (FT.TRANSFORMS)
IMG_SIZE = 224                             # 518 for DINOv2 at its native resolution (then run DINOv2 alone with its own OUT_ROOT)
print(json.dumps({k: v for k, v in FT.TIMM_IDS.items() if k in BACKBONES}, indent=1)); print(json.dumps(FT.TRANSFORMS, indent=1))


In [ ]:
# ---- 1. Determinism check: a fixed random input forwarded twice per backbone, hashed; a mismatch means nondeterministic kernels ----
import json
for b in BACKBONES:
    print(json.dumps(FT.determinism_check_timm(b, 'cuda')))


In [ ]:
# ---- 2. Extraction (idempotent; finished caches are skipped; re-run the cell after a session death) ------------------
# Expect roughly: CIFAR-100 6 min per backbone, ImageNet-R 4, CUB 2, OmniBenchmark 10 on a T4; about 1.5 GPU-hours in all.
import subprocess, sys, json
cmd = [sys.executable, 'replicate.py', 'features-timm', '--data-root', DATA_ROOT, '--out-root', OUT_ROOT,
       '--backbones', ','.join(BACKBONES), '--datasets', ','.join(DATASETS), '--batch-size', '128',
       '--transform', TRANSFORM, '--img-size', str(IMG_SIZE)] + (['--fp16'] if FP16 else [])
subprocess.run(cmd, check=False)
print(json.dumps(json.load(open(OUT_ROOT + '/MANIFEST.json')), indent=1)[:3000])


In [ ]:
# ---- 3. Readback: the fingerprint grid and Condition (S) on the recovered-protocol caches (minutes, CPU) -------------
import subprocess, sys, json, pathlib
RES = pathlib.Path('results/timm'); RES.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, 'replicate.py', '--features', OUT_ROOT, '--results', str(RES), 'fingerprint', '--seeds', '0'], check=True)
subprocess.run([sys.executable, 'replicate.py', '--features', OUT_ROOT, '--results', str(RES), 'condition-s', '--seeds', '0,1,2,3,4'], check=True)
(WORK / 'results' / 'timm').mkdir(parents=True, exist_ok=True)
for f in ('FINGERPRINT.md', 'FINGERPRINT.json', 'paperA_conditionS.json'):
    import shutil; shutil.copy(RES / f, WORK / 'results' / 'timm' / f)
print(open(RES / 'FINGERPRINT.md').read()[:6000])


In [ ]:
# ---- 4. Decision rules (FINGERPRINT_DECISION_TREE.md, section 3), applied to the ranking ------------------------------
import json, numpy as np
rep = json.load(open(RES / 'FINGERPRINT.json'))
rank = {k: (w, m, n) for k, w, m, n in rep['ranking']['gamma']}
def verdict(name, tol=0.10):
    w, m, n = rank.get(name, (np.inf, np.inf, 0)); return ('PASS' if w < tol else 'fail'), w, n
print('configs with a printed value:', sum(1 for c in rep['configs'].values() if c.get('printed_gamma')))
for name in ['lw_pooled (ours)', 'sample_pooled', 'zscore_lw', 'l2norm_lw', 'eq1_pair_lw', 'euclid_over_mean_var', 'euclid_over_lambda_max'] + [k for k in rank if k.startswith('ridge_abs')]:
    v, w, n = verdict(name); print(f'{name:28s} {v:5s} worst |log ratio| {w:.3f} over {n} configs')
best = rep['ranking']['gamma'][0]; print('\nbest gamma convention:', best)
lw = verdict('lw_pooled (ours)')
if lw[0] == 'PASS':
    print('LEAF: the recovered protocol with Ledoit-Wolf reproduces the printed gamma_min: PROTOCOL CARD = timm pooled pre-logits + dataset statistics + Ledoit-Wolf pooled covariance; regenerate tab:gamma-min from these caches (world (i)).')
elif best[1] < 0.10:
    print(f'LEAF: the recovered protocol reproduces the printed gamma_min under the estimator convention {best[0]!r}: PROTOCOL CARD = timm + that estimator; name it in app:gamma-computation; the Ledoit-Wolf column prints beside it.')
else:
    print('LEAF L7 (residual): no estimator convention on the recovered-protocol features brings all printed gamma_min within 10 percent. The printed table is not a convention applied to these features; the provenance framework (PROVENANCE_FRAMEWORK.md) and world (ii) of the abstract apply.')
nu_best = rep['ranking']['nu'][:5]; rhs_best = rep['ranking']['rhs'][:3]
print('\nbest nu cells (worst abs error):'); [print('  ', r) for r in nu_best]
print('best RHS cells (worst |log ratio|):'); [print('  ', r) for r in rhs_best]
print('nu verdict:', 'FORMULA FOUND' if nu_best and nu_best[0][1] <= 0.08 else 'no formula cell within 0.08 on every configuration (leaf M3)')
